# Linear Model Using PyTorch

First PyTorch model: a linear regression built with `nn.Sequential`, trained on synthetic data with a known ground-truth weight and bias. The goal is to learn PyTorch's core mechanics (tensors, autograd, `nn.Module`) by hand-rolling each piece before reaching for its built-in abstraction, then verify the trained model actually recovers the true weight/bias it was generated from.

## Phase 0 — Environment Setup

`deep_learning/` is its own isolated `uv` project (same pattern as `Week-1`/`Week-2`), so PyTorch gets added here without touching any other week's environment: `uv add torch` (already run — `torchvision`/`torchaudio` are skipped, they're for image/audio pipelines we don't need for tabular tensors).

Before writing any model code, confirm the install worked and check which compute device is actually available. This repo runs on a Mac, so the interesting device beyond `cpu` is Apple's Metal backend (`mps`) — checking for it now means every later block can write device-agnostic code (`.to(device)`) instead of hardcoding `cpu`.

In [ ]:
import torch

print(f"torch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using device: {device}")

**torch 2.14.0**, and **MPS is available** — so `device` resolves to `mps`, meaning tensor ops and model training below run on the Mac's GPU instead of the CPU. `torch.device("mps" if torch.backends.mps.is_available() else "cpu")` is the pattern used for the rest of the notebook: every tensor and model gets moved onto `device` explicitly rather than assuming CPU, so the same code would still work unchanged on a machine without MPS.

(A `UserWarning` about NumPy printed above too — `numpy` isn't installed in this project yet. That's expected; it gets added in Block 1 when synthetic data generation needs it, not before.)

**Phase 0 checkpoint reached: PyTorch is installed, imports cleanly, and MPS is confirmed available.** Next up is Block 1 — tensors.

## Block 1 — Tensors

A `torch.Tensor` is the same idea as a numpy array — an n-dimensional array of numbers — with two things bolted on that matter for deep learning: it can live on a GPU (`device`), and it can track the operations done to it for automatic differentiation (Block 2). Everything else — shape, dtype, indexing, broadcasting, matmul — works the same way you'd expect from numpy, and torch tensors convert to/from numpy arrays essentially for free.

`numpy` gets added to this project now (`uv add numpy`) since the round-trip below needs it, and it's also what generates the synthetic dataset in Block 4 — same convention as every other notebook in this repo.

In [ ]:
import numpy as np

# from a plain Python list
t_from_list = torch.tensor([1.0, 2.0, 3.0])

# from a numpy array
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
# we can cast the dtype of numpy array to float32 to match PyTorch's default dtype
t_from_numpy = torch.tensor(arr, dtype=torch.float32)

# built-in constructors
t_zeros = torch.zeros(2, 3)
t_randn = torch.randn(2, 3)

for name, t in [
    ("t_from_list", t_from_list),
    ("t_from_numpy", t_from_numpy),
    ("t_zeros", t_zeros),
    ("t_randn", t_randn),
]:
    print(f"{name:14s} shape={tuple(t.shape)}  dtype={t.dtype}")

In [ ]:
# elementwise ops + broadcasting
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print("a + b      =", a + b)
print("a * 2      =", a * 2)  # broadcasting a scalar
print("a * b      =", a * b)  # elementwise, not matmul

# matmul — the actual op behind y = Xw + b
X = torch.randn(4, 3)  # 4 samples, 3 features
w = torch.randn(3, 1)  # 3 weights, 1 output
y = X @ w  # (4,3) @ (3,1) -> (4,1)
print("\nX @ w shape:", tuple(y.shape))

In [ ]:
# numpy <-> tensor round trip, and moving a tensor onto `device`
original = np.array([1.0, 2.0, 3.0], dtype=np.float32)
as_tensor = torch.from_numpy(original)
back_to_numpy = as_tensor.numpy()

print("round-trip equal:", np.allclose(original, back_to_numpy))

t_on_device = t_randn.to(device)
print("t_randn device before:", t_randn.device)
print("t_on_device device after:", t_on_device.device)

Shapes are exactly what was asked for — `t_zeros`/`t_randn` are `(2, 3)`, `t_from_numpy` is `(2, 2)` matching the nested list it came from. All four dtypes come out `float32` — but only because `t_from_numpy` was built with an explicit `dtype=torch.float32` cast. Without that cast, `torch.tensor(arr)` would've inherited `float64` from numpy's default float dtype, while `t_from_list`/`t_zeros`/`t_randn` default to `float32` (torch's own default, used whenever the input is dtype-less Python data instead of an existing numpy/tensor dtype). This is exactly the mismatch that bites in Block 4: `nn.Linear`'s weights are `float32`, so numpy data has to be cast explicitly before it reaches a model — casting it here, right at creation, is the fix in practice.

`X @ w` on shapes `(4, 3) @ (3, 1)` produced `(4, 1)` — this is the literal operation `nn.Linear` does internally (`Xw + b`), just without the bias term yet. `a * b` above it stayed elementwise (`[10, 40, 90]`), not a dot product — `*` and `@` are different operators in torch, unlike numpy where `*` sometimes gets conflated with it for 1D arrays.

The numpy round-trip is exact (`np.allclose` → `True`) — tensors and numpy arrays share memory when both live on CPU, so this conversion is effectively free. And `.to(device)` moved `t_randn` from `cpu` to `mps:0` — this is the pattern used everywhere from here on to keep tensors and models on the same device (torch throws if you try to operate across devices).

**Self-check passed:** round-trip equality is `True`, all shapes and dtypes match expectations. Next up is Block 2 — autograd, hand-rolled.

## Block 2 — Autograd, Hand-Rolled

Any tensor created with `requires_grad=True` gets tracked: every operation done to it is recorded into a computation graph. Calling `.backward()` on a final scalar (like a loss) walks that graph backward and fills in `.grad` on every tracked tensor with the derivative of the scalar with respect to it — that's the entire mechanism behind `loss.backward()`, which gets used constantly from here on.

Before trusting it, verify it against a derivative computed by hand on a tiny scalar example. Then, before reaching for `nn.Module` and `torch.optim` in Block 3, write the gradient descent update loop by hand — so when those abstractions show up next, they're recognizably *the same three lines*, not new magic.

In [ ]:
# scalar toy example: pred = w * x, loss = (pred - y_true)**2
x = torch.tensor(3.0)
y_true = torch.tensor(10.0)
w = torch.tensor(1.0, requires_grad=True)

pred = w * x
loss = (pred - y_true) ** 2
loss.backward()

# hand-computed derivative: d(loss)/dw = 2 * (pred - y_true) * x
hand_grad = 2 * (pred.item() - y_true.item()) * x.item()

print(f"pred = {pred.item()}, loss = {loss.item()}")
print(f"w.grad (autograd) = {w.grad.item()}")
print(f"hand-computed grad = {hand_grad}")

Autograd's `w.grad` matches the hand-computed derivative exactly — trust established. Now the actual point of this block: write the gradient descent loop that `loss.backward()` + a manual update step performs, with no `nn.Module` or `torch.optim` involved, on a tiny toy dataset with a known true weight and bias.

In [ ]:
# tiny toy dataset with a KNOWN true weight/bias
w_true, b_true = 3.0, 5.0
x_np = np.linspace(-3, 3, 10).astype(np.float32)
y_np = (
    w_true * x_np
    + b_true
    + np.random.normal(0, 0.1, size=x_np.shape).astype(np.float32)
)

x_t = torch.from_numpy(x_np)
y_t = torch.from_numpy(y_np)

# start from scratch — no nn.Module, no torch.optim
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.01

for epoch in range(500):
    pred = w * x_t + b
    loss = ((pred - y_t) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}  loss={loss.item():.4f}")
        print(f"  w={w.item():.4f}  b={b.item():.4f}")

print(f"\nlearned w={w.item():.4f}  (true w={w_true})")
print(f"learned b={b.item():.4f}  (true b={b_true})")

Two things to check here. First, the scalar example: `pred = 3.0`, `loss = 49.0`, and **`w.grad` (autograd) = -42.0 matches the hand-computed grad = -42.0** exactly — autograd is doing precisely what the calculus says it should.

Second, the manual loop: loss starts at ~58 and drops to ~0.007 by epoch 400 (it can't reach exactly 0 — the dataset has noise with std 0.1, so a variance around `0.1² = 0.01` is the floor, not a bug). The learned `w` and `b` land close to `w_true=3.0` and `b_true=5.0` (this run: `w≈2.97`, `b≈5.07` — your exact numbers will differ slightly since the noise isn't seeded here, but they should land in the same neighborhood). There is no `nn.Module`, no `nn.MSELoss`, no `torch.optim` anywhere in this loop — just `requires_grad`, `.backward()`, and three lines of manual parameter update inside `torch.no_grad()`, plus zeroing `.grad` every step (skipping that would silently accumulate gradients across epochs and break convergence).

**Self-check passed:** learned `w`/`b` converged within ~0.1 of the true values. Block 3 replaces this exact loop with `nn.Linear` + `nn.MSELoss` + `torch.optim.SGD` — same mechanism, less boilerplate.

## Block 3 — `nn.Module`, `nn.Sequential`, Loss, Optimizer

Everything below is the exact same mechanism as Block 2, just with the boilerplate handled for you:

| Block 2 (manual) | Block 3 (`nn`) |
|---|---|
| `w`, `b` as separate tensors | `nn.Linear(1, 1)` holds both as `.weight`/`.bias` |
| `pred = w * x + b` | `model(x)` |
| `((pred - y) ** 2).mean()` | `nn.MSELoss()(pred, y)` |
| `w -= lr * w.grad` (manual update) | `optimizer.step()` |
| `w.grad.zero_()` | `optimizer.zero_grad()` |

`nn.Sequential(nn.Linear(1, 1))` — the "Sequential" model — chains layers in order; here there's only one layer, a single `nn.Linear` with 1 input feature and 1 output, which is exactly `y = w*x + b` internally, just with `w`/`b` managed for you and randomly initialized instead of starting at zero.

In [ ]:
import torch.nn as nn

model = nn.Sequential(nn.Linear(1, 1))
print(model)

for name, p in model.named_parameters():
    print(f"{name:20s} shape={tuple(p.shape)}  value={p.data}")

In [ ]:
# reuse Block 2's toy dataset — reshape to (N, 1) since nn.Linear expects that shape
X_demo = x_t.reshape(-1, 1)
Y_demo = y_t.reshape(-1, 1)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# forward pass BEFORE any training
pred_before = model(X_demo)
loss_before = loss_fn(pred_before, Y_demo)

# the exact three-line pattern used everywhere from here on
optimizer.zero_grad()
loss_before.backward()
optimizer.step()

# forward pass AFTER one single update step
pred_after = model(X_demo)
loss_after = loss_fn(pred_after, Y_demo)

print(f"loss before 1 step: {loss_before.item():.4f}")
print(f"loss after  1 step: {loss_after.item():.4f}")

`print(model)` shows `Sequential(Linear(in_features=1, out_features=1, bias=True))` — one layer, 1 input, 1 output, with a bias term included by default. `named_parameters()` shows it really is just a `weight` (shape `(1, 1)`) and a `bias` (shape `(1,)`) under the hood — the same two numbers as Block 2's `w` and `b`, except `nn.Linear` initializes them to small random values (here `weight≈-0.0075`, `bias≈0.54`) instead of starting at zero.

The one-step demo: loss goes from `53.42` (this run — yours will differ, random init isn't seeded) down to `47.91` after a *single* `optimizer.step()` call. That call did exactly what Block 2's `w -= lr * w.grad` / `b -= lr * b.grad` did by hand — `optimizer.zero_grad()` clears old gradients (same job as `.grad.zero_()`), `.backward()` computes new ones, `optimizer.step()` applies the update using those gradients and the optimizer's own rule (plain SGD here matches the manual math exactly).

**Self-check passed:** one step reduced the loss, confirming the `zero_grad → backward → step` pattern works correctly. Block 4 repeats this exact three-line pattern inside a training loop over many epochs — the actual linear regression project, on the real synthetic dataset.

## Block 4 — The Project: Linear Regression on Synthetic Data

This is the real thing, at proper scale: a full train/val split, a training loop over many epochs, and a check at the end of whether the model actually recovered the true weight and bias it was generated from.

Synthetic data is used deliberately here instead of `sklearn.datasets.make_regression` or a real dataset like California housing: with synthetic data **we already know the correct answer** (`w_true`, `b_true`), so "did training work?" has a concrete, checkable answer — compare what the model learned against the number that generated the data — instead of just "did the loss go down," which doesn't tell you if the model found the *right* relationship or just some relationship that fits the noise.

`numpy` generates the data and `sklearn.model_selection.train_test_split` does the train/val split — same libraries this repo already uses everywhere else, reused rather than hand-rolled.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

# same true w/b as Block 2's toy example, now with a proper-sized, noisier dataset
np.random.seed(42)
N = 200
X_all = np.random.uniform(-5, 5, size=(N, 1)).astype(np.float32)
y_all = (w_true * X_all + b_true + np.random.normal(0, 0.5, size=(N, 1))).astype(
    np.float32
)

print(f"X_all shape: {X_all.shape}, y_all shape: {y_all.shape}")

plt.figure(figsize=(6, 4))
plt.scatter(X_all, y_all, alpha=0.5, s=15)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Synthetic data: y = {w_true}*x + {b_true} + noise")
plt.show()

In [ ]:
print(X_all[:5])
print(y_all[:5])

In [ ]:
from sklearn.model_selection import train_test_split

X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

# already (N, 1) and float32 from generation above — no reshape/cast needed this time
X_train = torch.tensor(X_train_np, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train_np, dtype=torch.float32).to(device)
X_val = torch.tensor(X_val_np, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val_np, dtype=torch.float32).to(device)

print(f"train: {tuple(X_train.shape)}  val: {tuple(X_val.shape)}")

200 samples split 80/20 into train/val, both already `float32` and shape `(N, 1)` from how they were generated above (avoiding both the dtype gotcha from Block 1 and the reshape from Block 3 by getting the shape right at creation time). Both tensors are also moved `.to(device)` right away, and the model gets built on `device` too below — this is the first explicit "production practice" of the notebook: `torch.manual_seed(42)` right before building the model, so the random weight/bias initialization is reproducible run to run (contrast with Block 3's demo, which was deliberately left unseeded).

In [ ]:
torch.manual_seed(42)

model = nn.Sequential(nn.Linear(1, 1)).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

n_epochs = 300
train_losses = []
val_losses = []

for epoch in range(n_epochs):
    model.train()
    pred_train = model(X_train)
    loss_train = loss_fn(pred_train, y_train)

    optimizer.zero_grad()
    loss_train.backward()
    optimizer.step()

    model.eval()
    with torch.inference_mode():
        pred_val = model(X_val)
        loss_val = loss_fn(pred_val, y_val)

    train_losses.append(loss_train.item())
    val_losses.append(loss_val.item())

    if epoch % 50 == 0:
        print(
            f"epoch {epoch:4d}  train_loss={loss_train.item():.4f}  val_loss={loss_val.item():.4f}"
        )

print(f"\nfinal train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training vs validation loss")
plt.legend()
plt.show()

In [ ]:
learned_w = model[0].weight.item()
learned_b = model[0].bias.item()

print(f"learned w = {learned_w:.4f}   (true w = {w_true})")
print(f"learned b = {learned_b:.4f}   (true b = {b_true})")

**The payoff.** Training loss drops from `58.04` at epoch 0 to `0.2250` at epoch 300, and val loss tracks it closely the whole way (`44.99` → `0.2712`) — never spiking above train loss, which is what you'd expect on a genuinely linear relationship with plenty of data relative to the model's 2 parameters (no overfitting to chase here). The final loss floor (~0.23–0.27) roughly matches the noise variance baked into the data (`std=0.5` → variance `0.25`) — the model isn't failing to fit, it's hit the irreducible noise floor.

And the number that actually matters: **learned w = 3.0050 vs true w = 3.0**, **learned b = 5.0233 vs true b = 5.0**. The model was never told `w_true`/`b_true` — it only ever saw `(x, y)` pairs — and it recovered both to within about 0.5% and 2.3% of the true values. This is the concrete answer to "did training work": not just "loss went down," but "the model found the actual relationship the data was generated from."

**Self-check passed:** learned `w`/`b` within a small tolerance of true values, val loss tracked train loss without diverging. Block 5 reviews the production practices already used here (seeding, device placement, train/eval modes) and adds the one piece not yet shown — saving and reloading the trained model.

## Block 5 — Production Practices Review

Four practices were already used inline in Block 4, without calling them out by name. Naming them explicitly, and why each one matters:

- **Reproducible seeding** — `torch.manual_seed(42)` right before building the model. `nn.Linear`'s weight/bias start at random values; without a seed, every run gets a different random init and results aren't directly comparable run to run. With a seed, the same code always produces the same init.
- **Device-agnostic code** — `device` resolved once in Phase 0 (`mps` here), then every tensor and the model itself moved onto it with `.to(device)`. The code never hardcodes `"cpu"` or `"mps"` directly, so the exact same notebook would run unchanged on a machine without MPS.
- **`model.train()` / `model.eval()`** — set before the training forward pass and the validation forward pass respectively. For a bare `nn.Linear` this doesn't actually change any behavior (no dropout, no batchnorm), but layers that *do* behave differently in train vs. eval mode (dropout, batchnorm) will silently give wrong results if this switch is forgotten — worth building the habit now, before it's a layer that actually depends on it.
- **No gradient tracking during validation** — wrapping the validation forward pass in `torch.no_grad()` (the notebook currently uses `torch.inference_mode()`, a stricter/slightly faster variant of the same idea) avoids building a computation graph for a pass that's never going to call `.backward()` — saves memory and compute for no downside.

One practice not shown yet: **saving and reloading a trained model.** That's the rest of this block.

In [ ]:
# prove the device-agnostic claim: same code, forced onto CPU, same seed -> same result
cpu_device = torch.device("cpu")
torch.manual_seed(42)

model_cpu = nn.Sequential(nn.Linear(1, 1)).to(cpu_device)
loss_fn_cpu = nn.MSELoss()
optimizer_cpu = torch.optim.SGD(model_cpu.parameters(), lr=0.01)

X_train_cpu = X_train.to(cpu_device)
y_train_cpu = y_train.to(cpu_device)

for epoch in range(n_epochs):
    model_cpu.train()
    pred = model_cpu(X_train_cpu)
    loss = loss_fn_cpu(pred, y_train_cpu)
    optimizer_cpu.zero_grad()
    loss.backward()
    optimizer_cpu.step()

print(
    f"cpu-trained  w={model_cpu[0].weight.item():.4f}  b={model_cpu[0].bias.item():.4f}"
)
print(f"{device}-trained  w={learned_w:.4f}  b={learned_b:.4f}")

In [ ]:
# save the trained model's parameters
torch.save(model.state_dict(), "linear_model_state.pt")
print("saved to linear_model_state.pt")

# build a FRESH model — different random init — then load the saved weights into it
fresh_model = nn.Sequential(nn.Linear(1, 1)).to(device)
print(
    f"fresh model before loading: w={fresh_model[0].weight.item():.4f}  b={fresh_model[0].bias.item():.4f}"
)

fresh_model.load_state_dict(torch.load("linear_model_state.pt"))
fresh_model.eval()

with torch.no_grad():
    pred_original = model(X_val)
    pred_reloaded = fresh_model(X_val)

print(
    f"fresh model after loading:  w={fresh_model[0].weight.item():.4f}  b={fresh_model[0].bias.item():.4f}"
)
print("predictions match:", torch.allclose(pred_original, pred_reloaded))

Two things confirmed here, both exactly:

**Device-agnostic training really is device-agnostic.** Forcing the *identical* code onto `cpu` with the *same* seed (`42`) produced `w=3.0050, b=5.0233` — matching the `mps`-trained model to four decimal places. Same math, same result, regardless of which device actually ran it. This is what `.to(device)` throughout the notebook buys: no code changes needed to run on a machine without MPS.

**`state_dict()` round-trips exactly.** Before loading, `fresh_model` has a totally different random init (`w=-0.2343, b=0.9186` — a fresh, un-seeded `nn.Linear`). After `fresh_model.load_state_dict(torch.load(...))`, its weights become `w=3.0050, b=5.0233` — identical to the original trained `model`. And **`predictions match: True`** — `fresh_model` and `model` produce bit-for-bit identical predictions on the validation set. `state_dict()` only saves the learned numbers (weights/biases), not the model architecture or any Python code, so it's portable and doesn't depend on pickling the class definition — the standard way to persist a trained PyTorch model.

**Self-check passed:** cpu/mps parity confirmed, reload predictions match exactly. Block 6 wraps up with a results summary and what's deliberately deferred to a future notebook.

## Block 6 — Wrap-up

| quantity | true | learned |
|---|---|---|
| weight (w) | 3.0000 | 3.0050 |
| bias (b) | 5.0000 | 5.0233 |
| | | |
| final train loss | — | 0.2250 |
| final val loss | — | 0.2712 |

**One-sentence takeaway:** a single `nn.Linear` layer trained with plain SGD recovered the true generating weight and bias to within noise, confirming the `optimizer.zero_grad() → loss.backward() → optimizer.step()` loop from Block 3 is doing exactly the gradient descent hand-rolled in Block 2 — `nn.Module` and `torch.optim` are convenience, not a different mechanism.

**Next steps / deferred** — deliberately not covered in this notebook, candidates for a follow-up:
- `Dataset`/`DataLoader` classes for batching (this notebook fed the whole training set through the model every epoch — fine at 160 rows, not at scale)
- learning rate schedulers (the flat `lr=0.01` used throughout works here; larger models usually need it to change over training)
- experiment tracking / checkpointing strategies (saving *periodically* during training, not just once at the end)
- a GPU deep-dive beyond the basic `.to(device)` pattern used here (batch size tuning, memory management, mixed precision)
- a **classification** model next — logistic regression or a small MLP — as the natural next step beyond regression